Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 09 · Reading the lake directly (for users who already hold lake access)

> **For users who already hold lake access only.** This notebook reads the lake's files with `deltalake` (or `pyiceberg`), not through Drishti. It gets **no entitlements and no masks** (you see every row and column your storage rights allow), **no in-memory tables** (they are never on disk) and **no computed figures** (VaR vectors, model measures, pivots are computed by the server). Everyone else, and every governed use, reads through Flight (notebooks 01 to 08).

The lake configuration (`config/lakes/<lake>.yaml`, LAKE_CONFIG.md) says where each table is, its format and its partitions; this notebook finds the table through it, as every reader in Drishti does, and only reads.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** No token: this read goes to storage, with your own storage rights. `deltalake` reads Delta tables, `pyiceberg` Iceberg ones (`requirements.txt`). `DRISHTI_LAKE_CONFIG` names the lake file (default: the demo lake's).

In [ ]:
import os
import re
from pathlib import Path

import yaml

REPO = Path.cwd().resolve().parents[2]       # examples/clients/python -> the repository
LAKE_FILE = Path(os.environ.get("DRISHTI_LAKE_CONFIG", REPO / "config" / "lakes" / "demo.yaml"))
lake = yaml.safe_load(LAKE_FILE.read_text())
print("lake:", lake["lake"], "| format:", lake.get("format"), "| root:", lake["root"], "|", len(lake.get("tables", {})), "tables")

## Where the table is

The root may name an environment variable with a default (`${DRISHTI_DELTA_ROOT:./data/delta}`); a relative root is relative to the server's working folder (the repository, for the demo). A table is `<domain>.<kind>`, its folder `<domain>/<kind>` unless it says `path:`. Every table is partitioned by `business_date` first.

In [ ]:
def expand(text):
    """${NAME:default} as the server resolves it: the innermost first (a default may hold another placeholder); NAME is an
    environment variable or a setting (drishti.data.dir is also read as DRISHTI_DATA_DIR)."""
    inner = re.compile(r"\$\{([\w.\-]+)(?::([^${}]*))?\}")
    def value(m):
        name = m.group(1)
        return os.environ.get(name) or os.environ.get(re.sub(r"[.\-]", "_", name).upper()) or (m.group(2) or "")
    text = str(text)
    while True:
        expanded = inner.sub(value, text)
        if expanded == text:
            return expanded
        text = expanded

root = expand(lake["root"])
if "://" not in root and not Path(root).is_absolute():
    root = str((REPO / root).resolve())
TABLE = os.environ.get("DRISHTI_EXAMPLE_LAKE_TABLE", "trading.trade_endur")
entry = lake["tables"][TABLE] or {}
domain, kind = TABLE.split(".", 1)
location = f"{root.rstrip('/')}/{entry.get('path', f'{domain}/{kind}')}"
fmt = entry.get("format", lake.get("format", "delta"))
print(TABLE, "->", location, f"({fmt})")
print("partitions: business_date,", ", ".join(entry.get("partition-by", [])) or "-")
print("promoted:", ", ".join(entry.get("promote", [])) or "-")

## Open it, read only

Delta: `DeltaTable(location).to_pyarrow_dataset()`. Iceberg: the table's latest metadata file, opened as a `StaticTable` (path tables; a catalogue lake opens it through its catalogue instead).

In [ ]:
import pyarrow.dataset as ds

def open_dataset(location, fmt):
    if fmt == "iceberg":
        from pyiceberg.table import StaticTable
        metadata = sorted(Path(location, "metadata").glob("*.metadata.json"))[-1]
        return StaticTable.from_metadata(str(metadata)).scan().to_arrow()     # an Arrow table
    from deltalake import DeltaTable
    return DeltaTable(location).to_pyarrow_dataset()                           # a lazy dataset: nothing read yet

data = open_dataset(location, fmt)
print(data.schema)

## The latest business date, promoted columns only

Partition pruning on `business_date` reads one date's files; the promoted columns are typed columns, so the read touches only them.

In [ ]:
import pyarrow.compute as pc

table = data if not isinstance(data, ds.Dataset) else data.to_table(columns=["business_date"])
latest = pc.max(table.column("business_date")).as_py()
columns = [c for c in ["business_date"] + entry.get("promote", []) if c in data.schema.names]
if isinstance(data, ds.Dataset):
    rows = data.to_table(filter=ds.field("business_date") == latest, columns=columns)
else:
    rows = data.filter(pc.equal(data.column("business_date"), latest)).select(columns)
print("business date", latest, "|", rows.num_rows, "rows")
rows.slice(0, 10).to_pandas()

## What this read is not

- **Not governed:** no row rules, no masks, no audit in Drishti. Your storage rights are the only check.
- **Not complete:** in-memory tables (notebook 08) are not on disk.
- **Not the figures:** VaR, ES, a model's measures and pivot totals are the server's (notebooks 03 to 05).

To share what you find, read it through Flight, so the next person sees what their own access allows.